# Ground Truth 验证系统

从本地 Excel 数据计算指标，与后端 API 结果对比，验证后端计算逻辑正确性。

## 1. 导入依赖

In [1]:
from pathlib import Path
from datetime import datetime
import sys

sys.path.insert(0, str(Path.cwd().parent))

from ground_truth_validation.config import (
    DATA_DIR, BUILDING_ID, SYSTEM_ID, CALIBRATION_MONTH,
    EPSILON_TOLERANCE, BACKEND_URL, ALL_METRICS
)
from ground_truth_validation.excel_reader import ExcelReader
from ground_truth_validation.backend_client import BackendAPIClient
from ground_truth_validation.orchestrator import MetricOrchestrator
from ground_truth_validation.comparison import ComparisonEngine
from ground_truth_validation.report import ReportGenerator

## 2. 初始化组件

In [2]:
reader = ExcelReader(DATA_DIR)
backend = BackendAPIClient(BACKEND_URL)
orchestrator = MetricOrchestrator(reader)
comparator = ComparisonEngine(EPSILON_TOLERANCE)
reporter = ReportGenerator()

print("组件初始化完成")

组件初始化完成


## 3. 检查后端连接

In [3]:
if backend.health_check():
    print("后端 API 可访问")
else:
    print("后端 API 无法访问")

后端 API 可访问


## 4. 设置验证参数

In [4]:
time_start = datetime.fromisoformat(f"{CALIBRATION_MONTH}-01T00:00:00")
time_end = datetime.fromisoformat(f"{CALIBRATION_MONTH}-31T23:59:59")

print(f"时间范围: {time_start} 至 {time_end}")
print(f"建筑: {BUILDING_ID}, 系统: {SYSTEM_ID}")
print(f"待验证指标: {len(ALL_METRICS)} 个")

时间范围: 2025-07-01 00:00:00 至 2025-07-31 23:59:59
建筑: G11, 系统: 1
待验证指标: 27 个


## 5. 执行验证

In [5]:
results = []

for i, metric_name in enumerate(ALL_METRICS, 1):
    print(f"\n[{i}/{len(ALL_METRICS)}] {metric_name}")
    
    try:
        gt_result = orchestrator.calculate_metric(metric_name, time_start, time_end, BUILDING_ID, SYSTEM_ID)
        print(f"  GT: {gt_result.get('value')}")
    except Exception as e:
        print(f"  GT 错误: {e}")
        gt_result = {"value": None, "status": "error"}
    
    try:
        backend_result = backend.calculate_metric(metric_name, time_start, time_end, BUILDING_ID, SYSTEM_ID)
        print(f"  后端: {backend_result.get('value')}")
    except Exception as e:
        print(f"  后端错误: {e}")
        backend_result = {"value": None, "status": "error"}
    
    comparison = comparator.compare(gt_result, backend_result)
    
    results.append({
        "metric_name": metric_name,
        "gt_result": gt_result,
        "backend_result": backend_result,
        "comparison": comparison
    })
    
    status = "✅" if comparison['match'] else "❌"
    print(f"  {status}")


[1/27] 系统总电量
  GT: 5272552.94
  后端: None
  ❌

[2/27] 冷机能耗占比
  GT: None
  后端: None
  ✅

[3/27] 水泵能耗占比
  GT: None
  后端: None
  ✅

[4/27] 风机能耗占比
  GT: None
  后端: None
  ✅

[5/27] 冷冻水供水温度
  GT: None
  后端: None
  ✅

[6/27] 冷冻水回水温度
  GT: None
  后端: None
  ✅

[7/27] 冷却水供水温度
  GT: None
  后端: None
  ✅

[8/27] 冷却水回水温度
  GT: None
  后端: None
  ✅

[9/27] 冷冻水温差
  GT: None
  后端: None
  ✅

[10/27] 冷冻水流量
  GT: None
  后端: None
  ✅

[11/27] 冷却水流量
  GT: None
  后端: None
  ✅

[12/27] 制冷量
  GT: None
  后端: None
  ✅

[13/27] 冷机平均负载率
  GT: None
  后端: None
  ✅

[14/27] 冷机最大负载率
  GT: None
  后端: None
  ✅

[15/27] 冷机负载波动系数
  GT: None
  后端: None
  ✅

[16/27] 冷机COP
  GT: None
  后端: None
  ✅

[17/27] 制冷系统COP
  GT: None
  后端: None
  ✅

[18/27] 冷冻泵工作频率
  GT: None
  后端: None
  ✅

[19/27] 冷却泵工作频率
  GT: None
  后端: None
  ✅

[20/27] 冷冻泵能耗密度
  GT: None
  后端: None
  ✅

[21/27] 冷却泵能耗密度
  GT: None
  后端: None
  ✅

[22/27] 冷却水温差
  GT: None
  后端: None
  ✅

[23/27] 冷却塔风机功率
  GT: None
  后端: None
  ✅

[24/27] 冷却塔效率
  GT: None
  后端: 

## 6. 生成报告

In [ ]:
output_path = Path("validation_report.md")
reporter.generate(results, output_path, CALIBRATION_MONTH, BUILDING_ID, SYSTEM_ID)
print(f"📄 报告: {output_path.absolute()}")

## 7. 汇总

In [ ]:
total = len(results)
matched = sum(1 for r in results if r['comparison']['match'])

print(f"\n总数: {total}")
print(f"匹配: {matched} ({matched/total*100:.1f}%)")
print(f"不匹配: {total-matched} ({(total-matched)/total*100:.1f}%)")